In [13]:
import pandas as pd
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
import os
import sys
sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import load_sleep_data, write_to_hdf5_file
from tqdm import tqdm

### readme:
in previous steps, sleep stages, apnea predictions etc and breathing features are not necessarily computed in the same step.  
all of them, however, yield same type of h5 files.  
this could, should always take the most recent version of the 'individual' h5 files and combine it to a final/combined h5 file.  
directory of final h5 should always stay the same and all files within this directory should always contain same version.

In [14]:
cohort = 'icu'

output_dir = f'/media/ssd2/{cohort}_final_files/'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# breathing
dir_br = '/media/ssd2/icu_breathing_instability_index/'
files_br = os.listdir(dir_br)
files_br.sort()

# sleep staging
dir_ss = '/media/ssd2/icu_sleepstaging/'
files_ss = os.listdir(dir_ss)
files_ss.sort()

# apnea prediction, self-similarity, hypoxia burden
# APNEA: this code needs to be computed for 'full data', not only night, once breathing script is finished.
# ignore until then.
# dir_ap = '/media/ssd2/icu_sleep_combined/h5_step4'
# files_ap = os.listdir(dir_ap)
# files_ap.sort()

# first, check if same files exist in all those directories.


In [15]:
print('breathing:')
print(files_br[:3])
print(len(set(files_br)))
print('sleep staging:')
print(files_ss[:3])
print(len(set(files_ss)))

# print('apnea pred:')
# print(files_ap[:3])



breathing:
['icusleep_001.h5', 'icusleep_003.h5', 'icusleep_004.h5']
132
sleep staging:
['icusleep_001.h5', 'icusleep_002.h5', 'icusleep_003.h5']
133


In [16]:
assert len(set(files_br) - set(files_ss)) == 0 
# breathing code still runs, so only a subset of files is available for files_br. this check can be improved later #TODO 

In [11]:
# file_tmp = files_br[-1]

# # load data and check if index is equal.
# data_br, hdr = load_sleep_data(os.path.join(dir_br, file_tmp), idx_to_datetime=1)
# data_ss, hdr_ss = load_sleep_data(os.path.join(dir_ss, file_tmp), idx_to_datetime=1)
# assert all(data_br.index == data_ss.index)

# # merge data:
# columns_additional = [x for x in data_ss.columns if x not in data_br.columns]
# data_merged = data_br.join(data_ss[columns_additional])

# # and save:

# hdr['study_id'] = np.int32(hdr['study_id'])
# hdr['MRN'] = np.int32(hdr['MRN'])
# hdr['fs'] = np.int32(hdr['fs'])
# hdr['start_datetime'] = np.array([hdr['start_datetime'].year,hdr['start_datetime'].month,hdr['start_datetime'].day, hdr['start_datetime'].hour, hdr['start_datetime'].minute, hdr['start_datetime'].second, hdr['start_datetime'].microsecond])

# # write_to_hdf5_file(data_merged, os.path.join(output_dir, file_tmp), hdr=hdr, overwrite=True)




In [17]:
bm_signals = ['hr', 'spo2', 'art1d', 'art1s', 'art2d', 'art2s', 'nbpd', 'nbps']

for file_tmp in tqdm(files_br):
    
    if 1: 
        if os.path.exists(os.path.join(output_dir, file_tmp)):
            print('exists. continue')
            continue
    
    # load data and check if index is equal.
    data_br, hdr = load_sleep_data(os.path.join(dir_br, file_tmp), idx_to_datetime=1)
    data_ss, hdr_ss = load_sleep_data(os.path.join(dir_ss, file_tmp), idx_to_datetime=1)
    assert all(data_br.index == data_ss.index)

    # merge data:
    columns_additional = [x for x in data_ss.columns if x not in data_br.columns]
    data_merged = data_br.join(data_ss[columns_additional])
    
    # filter out large airgo signal
    airgo_features = [x for x in data_merged.columns if x not in bm_signals]
    data_merged.loc[data_merged.band>4030, airgo_features] = np.nan
    
    data_merged['stage_pred_vcomb1'] = data_merged['stage_pred_amendsumeffort'].values
    if 'stage_pred_activity1sec' in data_merged.columns:
        data_merged.loc[data_merged['stage_pred_activity1sec'] == 5, 'stage_pred_vcomb1'] = 5
        
    # and save:
    hdr['study_id'] = np.int32(hdr['study_id'])
    hdr['MRN'] = np.int32(hdr['MRN'])
    hdr['fs'] = np.int32(hdr['fs'])
    hdr['start_datetime'] = np.array([hdr['start_datetime'].year,hdr['start_datetime'].month,hdr['start_datetime'].day, hdr['start_datetime'].hour, hdr['start_datetime'].minute, hdr['start_datetime'].second, hdr['start_datetime'].microsecond])

    write_to_hdf5_file(data_merged, os.path.join(output_dir, file_tmp), hdr=hdr, overwrite=True)



  0%|          | 0/132 [00:00<?, ?it/s]

exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. contin


100%|██████████| 132/132 [01:01<00:00,  2.15it/s][A

exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue
exists. continue


In [7]:
file_tmp

'icusleep_128.h5'